In [4]:
# pip install databricks-connect

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

df = spark.table("samples.bakehouse.sales_transactions") 
df.show(20, truncate=False)


+-------------+----------+-----------+--------------------------+----------------------+--------+---------+----------+-------------+----------------+
|transactionID|customerID|franchiseID|dateTime                  |product               |quantity|unitPrice|totalPrice|paymentMethod|cardNumber      |
+-------------+----------+-----------+--------------------------+----------------------+--------+---------+----------+-------------+----------------+
|1002961      |2000253   |3000047    |2024-05-14 12:17:01.495952|Golden Gate Ginger    |8       |3        |24        |amex         |378154478982993 |
|1003007      |2000226   |3000047    |2024-05-10 23:10:10.239954|Austin Almond Biscotti|36      |3        |108       |mastercard   |2244626981238094|
|1003017      |2000108   |3000047    |2024-05-16 16:34:10.61372 |Austin Almond Biscotti|40      |3        |120       |mastercard   |2490570234487424|
|1003068      |2000173   |3000047    |2024-05-02 04:31:51.612094|Pearly Pies           |28      |3  

In [7]:
df.printSchema()
df.count()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



3333

In [10]:
from pyspark.sql import functions as F

renamed = df.select(
    F.col("quantity").alias("quantity"),
    F.col("unitPrice").alias("unit_price"),
    F.col("totalPrice").alias("total_price"),
)
renamed.describe("quantity", "unit_price", "total_price").show()

+-------+-----------------+----------+------------------+
|summary|         quantity|unit_price|       total_price|
+-------+-----------------+----------+------------------+
|  count|             3333|      3333|              3333|
|   mean|6.647764776477648|       3.0|19.943294329432945|
| stddev|6.544484308501296|       0.0|  19.6334529255051|
|    min|                1|         3|                 3|
|    max|               60|         3|               180|
+-------+-----------------+----------+------------------+



In [11]:
from pyspark.sql import functions as F

clean = (
    df
    .withColumn("txn_date", F.to_date(F.col("dateTime")))
    .filter(F.col("quantity") > 0)
    .filter(F.col("totalPrice").isNotNull())
)

In [12]:
daily_sales = (
    clean
    .groupBy("txn_date", "franchiseID")
    .agg(
        F.countDistinct("transactionID").alias("txn_count"),
        F.sum("totalPrice").alias("revenue"),
        F.sum("quantity").alias("units_sold"),
    )
    .orderBy("txn_date", "franchiseID")
)
daily_sales.show(20, truncate=False)

+----------+-----------+---------+-------+----------+
|txn_date  |franchiseID|txn_count|revenue|units_sold|
+----------+-----------+---------+-------+----------+
|2024-05-01|3000000    |2        |24     |8         |
|2024-05-01|3000001    |4        |72     |24        |
|2024-05-01|3000002    |5        |174    |58        |
|2024-05-01|3000003    |5        |84     |28        |
|2024-05-01|3000004    |6        |105    |35        |
|2024-05-01|3000005    |6        |99     |33        |
|2024-05-01|3000006    |5        |60     |20        |
|2024-05-01|3000007    |2        |54     |18        |
|2024-05-01|3000008    |6        |84     |28        |
|2024-05-01|3000009    |3        |30     |10        |
|2024-05-01|3000010    |5        |75     |25        |
|2024-05-01|3000011    |9        |153    |51        |
|2024-05-01|3000012    |4        |84     |28        |
|2024-05-01|3000013    |2        |33     |11        |
|2024-05-01|3000014    |4        |12     |4         |
|2024-05-01|3000015    |2   

In [13]:
masked = df.withColumn(
    "cardNumber",
    F.when(F.col("cardNumber").isNotNull(), F.lit("****")).otherwise(F.lit(None)),
)

In [14]:
from pyspark.sql.window import Window

w = Window.partitionBy("franchiseID").orderBy(F.desc("totalPrice"))
ranked = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") <= 5)